# Do ensembles of trees get worse when we add more trees

[Master Tree Ensemble Models course](https://www.trainindata.com/p/master-tree-ensemble-models)

We train a random forest, an AdaBoost and a gradient boosting machine with two thousand trees each, and we follow the performance in the test set after every tree that we add.

We do this first with data that we make ourselves, and then with five real datasets, and at the end we compare the performance in the train set with the performance in the test set, to see which of the three models overfits.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml, make_classification
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

N_ESTIMATORS = 2000
MODELS = ["random forest", "adaboost", "gradient boosting"]

### The experiment

The two boosting models give us the performance after each tree with `staged_predict_proba`.

The random forest does not, because its trees are independent and the order in which we add them is arbitrary, so we ask every tree for its prediction and we average the first tree, then the first two, then the first three, and so on, which is the forest that we would have obtained with that many trees.

In [ ]:
def prepare(X, y):
    X = pd.DataFrame(X).copy()
    for var in X.columns:
        if str(X[var].dtype) in ("object", "category", "str"):
            X[var] = X[var].astype("category").cat.codes
    X = X.astype(float).fillna(-1)
    y = pd.Series(np.asarray(y)).astype("category").cat.codes
    return X, y


def make_model(name):
    if name == "random forest":
        return RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=0, n_jobs=-1)
    if name == "adaboost":
        return AdaBoostClassifier(n_estimators=N_ESTIMATORS, random_state=0)
    return GradientBoostingClassifier(n_estimators=N_ESTIMATORS, random_state=0)

In [ ]:
def predictions_after_each_tree(model, X):
    if isinstance(model, RandomForestClassifier):
        trees = np.array([tree.predict_proba(X.to_numpy())[:, 1] for tree in model.estimators_])
        return np.cumsum(trees, axis=0) / np.arange(1, len(trees) + 1)[:, None]
    return [p[:, 1] for p in model.staged_predict_proba(X)]


def roc_auc_curves(name, X, y):
    X, y = prepare(X, y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=0
    )
    model = make_model(name)
    model.fit(X_train, y_train)

    curves = {}
    for set_name, features, target in [("train", X_train, y_train), ("test", X_test, y_test)]:
        curves[set_name] = [
            roc_auc_score(target, p) for p in predictions_after_each_tree(model, features)
        ]
    return pd.DataFrame(curves, index=range(1, len(curves["test"]) + 1))

### Data that we make ourselves

We make one dataset and we use it three times, once with clean labels, and twice with the labels of ten and twenty percent of the observations flipped, because the weakness that everybody mentions about boosting is that it chases the observations that it predicts badly, and a wrong label is an observation that no model can ever predict well.

In [ ]:
X, y = make_classification(n_samples=6000, n_features=20, n_informative=6, random_state=0)

targets = {"clean labels": y}
for noise in [0.1, 0.2]:
    rng = np.random.default_rng(0)
    targets[f"{int(noise * 100)}% wrong labels"] = np.where(rng.random(len(y)) < noise, 1 - y, y)

toy = {
    (name, labels): roc_auc_curves(name, X, target)
    for name in MODELS
    for labels, target in targets.items()
}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, name in zip(axes, MODELS):
    for labels in targets:
        curve = toy[(name, labels)]["test"]
        ax.plot(curve.index, curve, label=labels)
    ax.set_xlabel("Number of trees")
    ax.set_title(name)
    ax.legend(loc="lower right")
axes[0].set_ylabel("roc-auc in the test set")
fig.suptitle("Data that we made ourselves")
fig.tight_layout()
plt.show()

### Five real datasets

In [ ]:
datasets = {
    "credit-g": 31,
    "diabetes": 37,
    "spambase": 44,
    "phoneme": 1489,
    "churn": 40701,
}

data = {name: fetch_openml(data_id=data_id, as_frame=True) for name, data_id in datasets.items()}

real = {
    (name, dataset): roc_auc_curves(name, data[dataset].data, data[dataset].target)
    for name in MODELS
    for dataset in datasets
}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, name in zip(axes, MODELS):
    for dataset in datasets:
        curve = real[(name, dataset)]["test"]
        ax.plot(curve.index, curve, label=dataset)
    ax.set_xlabel("Number of trees")
    ax.set_title(name)
    ax.legend(loc="lower right")
axes[0].set_ylabel("roc-auc in the test set")
fig.suptitle("Five real datasets")
fig.tight_layout()
plt.show()

In [ ]:
summary = {
    (name, dataset): {
        "best test": curve["test"].max(),
        "best at tree": curve["test"].idxmax(),
        "final test": curve["test"].iloc[-1],
        "lost from best %": 100
        * (curve["test"].max() - curve["test"].iloc[-1])
        / curve["test"].max(),
        "final train": curve["train"].iloc[-1],
    }
    for (name, dataset), curve in real.items()
}

summary = pd.DataFrame(summary).T
summary.index.names = ["model", "dataset"]
summary.round(4)

### Do the models overfit

Now we plot the performance in the train set next to the performance in the test set, one row per model and one column per dataset.

In [ ]:
fig, axes = plt.subplots(len(MODELS), len(datasets), figsize=(18, 9), sharex=True)
for row, name in zip(axes, MODELS):
    for ax, dataset in zip(row, datasets):
        curve = real[(name, dataset)]
        ax.plot(curve.index, curve["train"], label="train")
        ax.plot(curve.index, curve["test"], label="test")
        ax.set_ylim(0.5, 1.02)
        ax.set_title(f"{name}, {dataset}", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("Number of trees")
for row in axes:
    row[0].set_ylabel("roc-auc")
axes[0][0].legend(loc="lower right")
fig.tight_layout()
plt.show()